In [1]:
import pandas as pd
import numpy as np

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
# Load the weather dataset

df = pd.read_parquet(
    "../data/raw_data/daily_weather.parquet"
)

print("Dataset loaded successfully!")
print("Dataset Shape:", df.shape)

Dataset loaded successfully!
Dataset Shape: (27635763, 14)


In [3]:
# Display dataset information

print("Dataset Information:")
df.info()

Dataset Information:
<class 'pandas.DataFrame'>
Index: 27635763 entries, 0 to 24220
Data columns (total 14 columns):
 #   Column                  Dtype         
---  ------                  -----         
 0   station_id              category      
 1   city_name               category      
 2   date                    datetime64[us]
 3   season                  category      
 4   avg_temp_c              float64       
 5   min_temp_c              float64       
 6   max_temp_c              float64       
 7   precipitation_mm        float64       
 8   snow_depth_mm           float64       
 9   avg_wind_dir_deg        float64       
 10  avg_wind_speed_kmh      float64       
 11  peak_wind_gust_kmh      float64       
 12  avg_sea_level_pres_hpa  float64       
 13  sunshine_total_min      float64       
dtypes: category(3), datetime64[us](1), float64(10)
memory usage: 2.6 GB


In [4]:
# Select columns required for the project

selected_columns = [
    "city_name",
    "date",
    "season",
    "precipitation_mm",
    "avg_wind_speed_kmh",
    "avg_sea_level_pres_hpa",
    "avg_temp_c"
]

model_df = df[selected_columns].copy()

print("Selected Dataset Shape:", model_df.shape)
model_df.head()

Selected Dataset Shape: (27635763, 7)


,city_name,date,season,precipitation_mm,avg_wind_speed_kmh,avg_sea_level_pres_hpa,avg_temp_c
0,Asadabad,1957-07-01,Summer,0.0,NaN,NaN,27.0
1,Asadabad,1957-07-02,Summer,0.0,NaN,NaN,22.8
2,Asadabad,1957-07-03,Summer,1.0,NaN,NaN,24.3
3,Asadabad,1957-07-04,Summer,4.1,NaN,NaN,26.6
4,Asadabad,1957-07-05,Summer,0.0,NaN,NaN,30.8


In [5]:
# Create a sample for model training

model_sample = model_df.sample(
    n=500000,
    random_state=42
).copy()

print("Sample Dataset Shape:", model_sample.shape)

Sample Dataset Shape: (500000, 7)


In [6]:
# Check missing values

missing_values = model_sample.isnull().sum()

print("Missing Values in Each Column:")
print(missing_values)

Missing Values in Each Column:
city_name                    262
date                           0
season                         0
precipitation_mm          120626
avg_wind_speed_kmh        404300
avg_sea_level_pres_hpa    427140
avg_temp_c                112534
dtype: int64


In [7]:
# Calculate missing-value percentage

missing_percentage = (
    model_sample.isnull().sum() / len(model_sample)
) * 100

print("Missing Value Percentage:")
print(missing_percentage.round(2))

Missing Value Percentage:
city_name                  0.05
date                       0.00
season                     0.00
precipitation_mm          24.13
avg_wind_speed_kmh        80.86
avg_sea_level_pres_hpa    85.43
avg_temp_c                22.51
dtype: float64


In [8]:
# Remove rows with missing city or target temperature

model_sample = model_sample.dropna(
    subset=["city_name", "avg_temp_c"]
).copy()

# Numerical columns
numeric_columns = [
    "precipitation_mm",
    "avg_wind_speed_kmh",
    "avg_sea_level_pres_hpa"
]

# Fill missing numerical values with median
for column in numeric_columns:
    model_sample[column] = model_sample[column].fillna(
        model_sample[column].median()
    )

print("Missing values handled successfully!")

print("\nRemaining Missing Values:")
print(model_sample.isnull().sum())

Missing values handled successfully!

Remaining Missing Values:
city_name                 0
date                      0
season                    0
precipitation_mm          0
avg_wind_speed_kmh        0
avg_sea_level_pres_hpa    0
avg_temp_c                0
dtype: int64


In [9]:
# Create date-based features

model_sample["year"] = model_sample["date"].dt.year
model_sample["month"] = model_sample["date"].dt.month
model_sample["day"] = model_sample["date"].dt.day
model_sample["day_of_year"] = model_sample["date"].dt.dayofyear

# Remove original date column

model_sample = model_sample.drop(
    columns=["date"]
)

print("Date features created successfully!")

model_sample.head()

Date features created successfully!


,city_name,season,precipitation_mm,avg_wind_speed_kmh,avg_sea_level_pres_hpa,avg_temp_c,year,month,day,day_of_year
1503,Kütahya,Summer,0.0,11.0,1014.7,25.7,2008,7,27,209
23578,Sion,Summer,4.5,11.0,1014.7,14.1,1965,7,22,203
8609,Damanhur,Winter,0.3,11.0,1014.7,12.3,1989,2,1,32
17783,Chihuahua,Summer,0.0,9.6,1009.2,29.0,2019,7,23,204
2594,Luanda,Autumn,0.0,11.0,1014.7,26.4,1968,5,1,122


In [10]:
# Check duplicate rows

duplicate_count = model_sample.duplicated().sum()

print("Duplicate Rows:", duplicate_count)

Duplicate Rows: 3


In [11]:
# Separate features and target

X = model_sample.drop(
    columns=["avg_temp_c"]
)

y = model_sample["avg_temp_c"]

print("Features (X) Shape:", X.shape)
print("Target (y) Shape:", y.shape)

print("\nFeature Columns:")
print(X.columns.tolist())

Features (X) Shape: (387204, 9)
Target (y) Shape: (387204,)

Feature Columns:
['city_name', 'season', 'precipitation_mm', 'avg_wind_speed_kmh', 'avg_sea_level_pres_hpa', 'year', 'month', 'day', 'day_of_year']


In [12]:
from sklearn.model_selection import train_test_split

# Split data into training and testing sets

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training Features Shape:", X_train.shape)
print("Testing Features Shape:", X_test.shape)

print("Training Target Shape:", y_train.shape)
print("Testing Target Shape:", y_test.shape)

Training Features Shape: (309763, 9)
Testing Features Shape: (77441, 9)
Training Target Shape: (309763,)
Testing Target Shape: (77441,)


In [13]:
# Encode categorical columns

X_train_encoded = pd.get_dummies(
    X_train,
    columns=["city_name", "season"],
    dtype=int
)

X_test_encoded = pd.get_dummies(
    X_test,
    columns=["city_name", "season"],
    dtype=int
)

# Ensure both datasets have the same columns

X_test_encoded = X_test_encoded.reindex(
    columns=X_train_encoded.columns,
    fill_value=0
)

print("Encoding completed successfully!")

print("Encoded Training Shape:", X_train_encoded.shape)
print("Encoded Testing Shape:", X_test_encoded.shape)

Encoding completed successfully!
Encoded Training Shape: (309763, 1245)
Encoded Testing Shape: (77441, 1245)


In [14]:
# Save processed data

X_train_encoded.to_csv(
    "../data/processed_data/X_train_encoded.csv",
    index=False
)

X_test_encoded.to_csv(
    "../data/processed_data/X_test_encoded.csv",
    index=False
)

y_train.to_csv(
    "../data/processed_data/y_train.csv",
    index=False
)

y_test.to_csv(
    "../data/processed_data/y_test.csv",
    index=False
)

print("Processed data saved successfully!")

Processed data saved successfully!
